# Cardiovascular Disease: Logistic Regression Modelling

This notebook builds and evaluates a binary logistic regression model to predict whether cardiovascular disease is present.

Research question: **To what extent can routine health indicators be used to predict whether an individual has cardiovascular disease?**

Use case: a cardiovascular risk-screening prototype. The model is intended to support prioritisation and communication, not medical diagnosis.

The modelling goal is to build a strong, explainable model while avoiding data leakage and overclaiming.

## Notebook Outputs

When run successfully, this notebook creates modelling evidence in:

- `project_1_cardiovascular_disease/04_outputs/tables/`
- `project_1_cardiovascular_disease/04_outputs/charts/`

Main outputs:

- candidate model comparison table
- selected threshold scan table
- final test metrics table
- final confusion matrix
- ROC curve and ROC-AUC
- precision-recall curve
- probability distribution chart
- feature effect table and chart

The app itself is not built in this notebook. This notebook prepares the modelling evidence and selected model specification for a later app prototype.

## 1. Setup

This cell imports the required libraries, sets paths, and creates output folders.

In [ ]:
import os
import tempfile
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'm5_matplotlib_cache'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)

RANDOM_STATE = 42


def find_workspace_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'project_1_cardiovascular_disease').exists():
            return candidate
    raise FileNotFoundError('Could not find project_1_cardiovascular_disease from the current working directory.')


WORKSPACE_ROOT = find_workspace_root()
PROJECT_DIR = WORKSPACE_ROOT / 'project_1_cardiovascular_disease'
CLEANED_DATA_PATH = PROJECT_DIR / '01_data' / 'processed' / 'cardio_cleaned.csv'
CHARTS_DIR = PROJECT_DIR / '04_outputs' / 'charts'
TABLES_DIR = PROJECT_DIR / '04_outputs' / 'tables'

for folder in [CHARTS_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Workspace root: {WORKSPACE_ROOT}')
print(f'Cleaned dataset path: {CLEANED_DATA_PATH}')
print(f'Charts folder: {CHARTS_DIR}')
print(f'Tables folder: {TABLES_DIR}')

## 2. Load the Cleaned Dataset

This notebook starts from the cleaned dataset created in `02_data_cleaning_and_eda.ipynb`.

In [ ]:
if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f'Cleaned dataset not found: {CLEANED_DATA_PATH}')

cardio_df = pd.read_csv(CLEANED_DATA_PATH)

required_columns = [
    'age_years', 'gender', 'height_cm', 'weight_kg', 'bmi',
    'ap_hi', 'ap_lo', 'pulse_pressure', 'cholesterol', 'gluc',
    'smoke', 'alco', 'active', 'age_band', 'bmi_category',
    'systolic_bp_category', 'diastolic_bp_category', 'cardio'
]

missing_columns = sorted(set(required_columns) - set(cardio_df.columns))
if missing_columns:
    raise ValueError(f'Missing required modelling columns: {missing_columns}')

print(f'Cleaned dataset shape: {cardio_df.shape[0]:,} rows x {cardio_df.shape[1]:,} columns')
display(cardio_df.head())

target_check = (
    cardio_df['cardio']
    .value_counts(normalize=False)
    .rename_axis('cardio')
    .reset_index(name='records')
    .sort_values('cardio')
)
target_check['share'] = target_check['records'] / len(cardio_df)
display(target_check)

## 3. Modelling Feature Engineering

The cleaning notebook created core fields such as age in years, BMI, and blood pressure bands.

This cell adds modelling-only fields that are still based on routine health indicators:

- mean arterial pressure: approximate average pressure in the arteries during one heartbeat
- blood pressure ratio: systolic divided by diastolic pressure
- age decade bands
- pulse pressure bands

These are derived only from predictor variables, so they do not leak the target.

In [ ]:
def add_modelling_features(df: pd.DataFrame) -> pd.DataFrame:
    output = df.copy()
    output['mean_arterial_pressure'] = output['ap_lo'] + (output['pulse_pressure'] / 3)
    output['bp_ratio'] = output['ap_hi'] / output['ap_lo']
    output['age_decade'] = pd.cut(
        output['age_years'],
        bins=[0, 40, 45, 50, 55, 60, 120],
        labels=['<40', '40-44', '45-49', '50-54', '55-59', '60+'],
        right=False,
    )
    output['pulse_pressure_band'] = pd.cut(
        output['pulse_pressure'],
        bins=[-np.inf, 40, 60, 80, np.inf],
        labels=['<40', '40-59', '60-79', '80+'],
        right=False,
    )
    return output


model_df = add_modelling_features(cardio_df)

modelling_feature_checks = model_df[
    ['mean_arterial_pressure', 'bp_ratio', 'age_decade', 'pulse_pressure_band']
].isna().sum().rename('missing_values').to_frame()

display(modelling_feature_checks)
display(model_df[['age_years', 'age_decade', 'ap_hi', 'ap_lo', 'pulse_pressure', 'pulse_pressure_band', 'mean_arterial_pressure', 'bp_ratio']].head())

## 4. Train, Validation, and Test Split

The split protects against overclaiming:

- training set: fit and tune model candidates
- validation set: choose the model and probability threshold
- test set: final evidence only, used once after selection

The split is stratified so the target balance is similar across all sets.

In [ ]:
y = model_df['cardio']

train_valid_df, test_df, y_train_valid, y_test = train_test_split(
    model_df,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

train_df, valid_df, y_train, y_valid = train_test_split(
    train_valid_df,
    y_train_valid,
    test_size=0.25,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame([
    {'split': 'Train', 'records': len(train_df), 'cardio_rate': y_train.mean()},
    {'split': 'Validation', 'records': len(valid_df), 'cardio_rate': y_valid.mean()},
    {'split': 'Test', 'records': len(test_df), 'cardio_rate': y_test.mean()},
])

split_summary.to_csv(TABLES_DIR / 'model_00_train_validation_test_split_summary.csv', index=False)
display(split_summary)

## 5. Candidate Models

All candidates are logistic regression models. They differ in feature representation, not algorithm family.

This keeps the project focused while still testing whether stronger feature engineering improves performance.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_logistic_pipeline(numeric_features, categorical_features, use_numeric_interactions=False):
    numeric_steps = [
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]
    if use_numeric_interactions:
        numeric_steps.append(('poly', PolynomialFeatures(degree=2, include_bias=False)))

    transformers = []
    if numeric_features:
        transformers.append(('numeric', Pipeline(numeric_steps), numeric_features))
    if categorical_features:
        transformers.append(('categorical', make_one_hot_encoder(), categorical_features))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    return Pipeline([
        ('preprocess', preprocessor),
        ('model', LogisticRegression(max_iter=4000, solver='lbfgs')),
    ])


candidate_models = {
    'baseline_original': {
        'description': 'Original routine indicators with numeric scaling.',
        'numeric_features': ['age_years', 'height_cm', 'weight_kg', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'gender'],
        'categorical_features': [],
        'use_numeric_interactions': False,
    },
    'clinical_numeric': {
        'description': 'Engineered continuous clinical indicators plus coded risk fields.',
        'numeric_features': ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure', 'bp_ratio'],
        'categorical_features': ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active'],
        'use_numeric_interactions': False,
    },
    'clinical_bands': {
        'description': 'Continuous health indicators plus interpretable clinical-style bands.',
        'numeric_features': ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure'],
        'categorical_features': [
            'gender', 'age_band', 'age_decade', 'bmi_category',
            'systolic_bp_category', 'diastolic_bp_category', 'pulse_pressure_band',
            'cholesterol', 'gluc', 'smoke', 'alco', 'active'
        ],
        'use_numeric_interactions': False,
    },
    'clinical_bands_numeric_interactions': {
        'description': 'Clinical bands plus second-order numeric terms for non-linear patterns.',
        'numeric_features': ['age_years', 'bmi', 'ap_hi', 'ap_lo', 'pulse_pressure', 'mean_arterial_pressure'],
        'categorical_features': [
            'gender', 'age_decade', 'bmi_category',
            'systolic_bp_category', 'diastolic_bp_category', 'pulse_pressure_band',
            'cholesterol', 'gluc', 'smoke', 'alco', 'active'
        ],
        'use_numeric_interactions': True,
    },
}

candidate_overview = pd.DataFrame([
    {
        'candidate': name,
        'description': config['description'],
        'numeric_feature_count': len(config['numeric_features']),
        'categorical_feature_count': len(config['categorical_features']),
        'uses_numeric_interactions': config['use_numeric_interactions'],
    }
    for name, config in candidate_models.items()
])

display(candidate_overview)

## 6. Threshold Selection Rule

The positive class is `cardio = 1`, meaning cardiovascular disease is present.

For the screening-support use case, a false negative is more serious than a false positive because it means a person with cardiovascular disease is missed by the model.

Threshold selection rule used on the validation set:

1. only consider thresholds with validation recall at least `0.80`
2. require validation precision at least `0.65`
3. require validation accuracy at least `0.70`
4. within those options, choose the threshold with the highest F1 score

This keeps recall high while avoiding an excessive false-positive rate.

In [ ]:
THRESHOLDS = np.round(np.arange(0.30, 0.61, 0.01), 2)
MIN_VALIDATION_RECALL = 0.80
MIN_VALIDATION_PRECISION = 0.65
MIN_VALIDATION_ACCURACY = 0.70


def calculate_classification_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    return {
        'threshold': float(threshold),
        'accuracy': accuracy_score(y_true, predictions),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'f1': f1_score(y_true, predictions, zero_division=0),
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
    }


def select_threshold(threshold_scan):
    qualified = threshold_scan[
        (threshold_scan['recall'] >= MIN_VALIDATION_RECALL)
        & (threshold_scan['precision'] >= MIN_VALIDATION_PRECISION)
        & (threshold_scan['accuracy'] >= MIN_VALIDATION_ACCURACY)
    ].copy()

    if qualified.empty:
        selected = threshold_scan.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0].copy()
        selected['threshold_rule'] = 'Fallback: highest validation F1 because no threshold met all constraints'
        return selected

    selected = qualified.sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0].copy()
    selected['threshold_rule'] = 'Selected from thresholds meeting recall, precision, and accuracy constraints'
    return selected


print('Threshold rule ready.')

## 7. Train and Compare Candidate Models

Each candidate is tuned on the training set using cross-validated ROC-AUC. The validation set is then used to select a threshold and compare candidates.

The test set is not used here.

In [ ]:
param_grid = {
    'model__C': [0.1, 1.0, 3.0],
    'model__class_weight': [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

candidate_rows = []
threshold_scan_rows = []
best_candidate_estimators = {}

for candidate_name, config in candidate_models.items():
    pipeline = build_logistic_pipeline(
        numeric_features=config['numeric_features'],
        categorical_features=config['categorical_features'],
        use_numeric_interactions=config['use_numeric_interactions'],
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=cv,
        n_jobs=None,
        refit=True,
    )
    grid_search.fit(train_df, y_train)

    best_estimator = grid_search.best_estimator_
    best_candidate_estimators[candidate_name] = best_estimator

    validation_probabilities = best_estimator.predict_proba(valid_df)[:, 1]
    validation_auc = roc_auc_score(y_valid, validation_probabilities)
    validation_average_precision = average_precision_score(y_valid, validation_probabilities)
    validation_brier = brier_score_loss(y_valid, validation_probabilities)

    candidate_threshold_scan = pd.DataFrame([
        calculate_classification_metrics(y_valid, validation_probabilities, threshold)
        for threshold in THRESHOLDS
    ])
    candidate_threshold_scan.insert(0, 'candidate', candidate_name)
    threshold_scan_rows.append(candidate_threshold_scan)

    selected_threshold_row = select_threshold(candidate_threshold_scan).to_dict()

    candidate_rows.append({
        'candidate': candidate_name,
        'description': config['description'],
        'cv_roc_auc_mean': grid_search.best_score_,
        'validation_roc_auc': validation_auc,
        'validation_average_precision': validation_average_precision,
        'validation_brier_score': validation_brier,
        'best_model_c': grid_search.best_params_['model__C'],
        'best_class_weight': str(grid_search.best_params_['model__class_weight']),
        'selected_threshold': selected_threshold_row['threshold'],
        'validation_accuracy': selected_threshold_row['accuracy'],
        'validation_precision': selected_threshold_row['precision'],
        'validation_recall': selected_threshold_row['recall'],
        'validation_f1': selected_threshold_row['f1'],
        'validation_false_negative': selected_threshold_row['false_negative'],
        'validation_false_positive': selected_threshold_row['false_positive'],
        'threshold_rule': selected_threshold_row['threshold_rule'],
    })

candidate_comparison = pd.DataFrame(candidate_rows)
all_threshold_scans = pd.concat(threshold_scan_rows, ignore_index=True)

candidate_comparison = candidate_comparison.sort_values(
    ['validation_recall', 'validation_f1', 'validation_roc_auc'],
    ascending=False,
).reset_index(drop=True)

candidate_comparison.to_csv(TABLES_DIR / 'model_01_candidate_comparison_validation.csv', index=False)
all_threshold_scans.to_csv(TABLES_DIR / 'model_02_all_candidate_threshold_scans_validation.csv', index=False)

rounded_candidate_comparison = candidate_comparison.copy()
for column in [
    'cv_roc_auc_mean', 'validation_roc_auc', 'validation_average_precision',
    'validation_brier_score', 'selected_threshold', 'validation_accuracy',
    'validation_precision', 'validation_recall', 'validation_f1'
]:
    rounded_candidate_comparison[column] = rounded_candidate_comparison[column].round(4)

display(rounded_candidate_comparison)
print(f'Saved candidate comparison to: {TABLES_DIR / "model_01_candidate_comparison_validation.csv"}')

## 8. Select the Final Model Specification

The selected model is chosen from validation results, prioritising recall first because this is a screening-support use case.

If two candidates are close, the simpler and more app-friendly model is preferred.

In [ ]:
selected_candidate_name = candidate_comparison.iloc[0]['candidate']
selected_candidate_config = candidate_models[selected_candidate_name]
selected_threshold = float(candidate_comparison.iloc[0]['selected_threshold'])
selected_best_c = float(candidate_comparison.iloc[0]['best_model_c'])
selected_class_weight_text = candidate_comparison.iloc[0]['best_class_weight']
selected_class_weight = None if selected_class_weight_text == 'None' else selected_class_weight_text

selected_model_summary = pd.DataFrame([
    {
        'selected_candidate': selected_candidate_name,
        'selected_threshold': selected_threshold,
        'best_model_c': selected_best_c,
        'class_weight': selected_class_weight_text,
        'selection_reason': 'Highest validation recall among candidates after threshold constraints, with strong F1 and ROC-AUC.',
        'model_family': 'Binary logistic regression',
        'positive_class': 'cardio = 1, cardiovascular disease present',
    }
])

selected_model_summary.to_csv(TABLES_DIR / 'model_03_selected_model_summary.csv', index=False)

display(selected_model_summary)
print(f'Selected model: {selected_candidate_name}')
print(f'Selected threshold: {selected_threshold:.2f}')

## 9. Validation Threshold Scan for the Selected Model

This chart shows why the selected threshold is appropriate for the screening-support use case.

In [ ]:
selected_threshold_scan = all_threshold_scans[
    all_threshold_scans['candidate'] == selected_candidate_name
].copy()
selected_threshold_scan.to_csv(TABLES_DIR / 'model_04_selected_threshold_scan_validation.csv', index=False)

display(selected_threshold_scan.round(4).head())
display(selected_threshold_scan.round(4).tail())

plt.rcParams.update({
    'figure.figsize': (9, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.titlesize': 13,
    'font.size': 10,
})

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(selected_threshold_scan['threshold'], selected_threshold_scan['accuracy'], label='Accuracy', linewidth=2)
ax.plot(selected_threshold_scan['threshold'], selected_threshold_scan['precision'], label='Precision', linewidth=2)
ax.plot(selected_threshold_scan['threshold'], selected_threshold_scan['recall'], label='Recall', linewidth=2)
ax.plot(selected_threshold_scan['threshold'], selected_threshold_scan['f1'], label='F1 score', linewidth=2)
ax.axvline(selected_threshold, color='#F26419', linestyle='--', linewidth=1.8, label=f'Selected threshold: {selected_threshold:.2f}')
ax.set_title('Validation metric trade-off by probability threshold')
ax.set_xlabel('Probability threshold')
ax.set_ylabel('Metric value')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0.45, 1.0)
ax.grid(axis='y', color='#D9DEE7', linewidth=0.8)
ax.legend(loc='lower left')

threshold_chart_path = CHARTS_DIR / 'model_01_validation_threshold_tradeoff.png'
fig.tight_layout()
fig.savefig(threshold_chart_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved chart: {threshold_chart_path}')

## 10. Final Test Evaluation

The final selected model is trained on the combined train and validation data, then evaluated once on the untouched test set.

This is the evidence result for the project.

In [ ]:
final_model = build_logistic_pipeline(
    numeric_features=selected_candidate_config['numeric_features'],
    categorical_features=selected_candidate_config['categorical_features'],
    use_numeric_interactions=selected_candidate_config['use_numeric_interactions'],
)
final_model.set_params(
    model__C=selected_best_c,
    model__class_weight=selected_class_weight,
)

final_model.fit(train_valid_df, y_train_valid)

test_probabilities = final_model.predict_proba(test_df)[:, 1]
test_predictions = (test_probabilities >= selected_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_predictions).ravel()

final_test_metrics = pd.DataFrame([
    {
        'selected_candidate': selected_candidate_name,
        'threshold': selected_threshold,
        'accuracy': accuracy_score(y_test, test_predictions),
        'precision': precision_score(y_test, test_predictions, zero_division=0),
        'recall': recall_score(y_test, test_predictions, zero_division=0),
        'f1': f1_score(y_test, test_predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_test, test_probabilities),
        'average_precision': average_precision_score(y_test, test_probabilities),
        'brier_score': brier_score_loss(y_test, test_probabilities),
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
        'test_records': len(test_df),
    }
])

final_test_metrics.to_csv(TABLES_DIR / 'model_05_final_test_metrics.csv', index=False)

rounded_final_test_metrics = final_test_metrics.copy()
for column in ['threshold', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'average_precision', 'brier_score']:
    rounded_final_test_metrics[column] = rounded_final_test_metrics[column].round(4)

display(rounded_final_test_metrics)
print(f'Saved final test metrics to: {TABLES_DIR / "model_05_final_test_metrics.csv"}')

## 11. Confusion Matrix

For this use case, the most serious error is a **false negative**: a person with cardiovascular disease predicted as not having cardiovascular disease.

In [ ]:
confusion_df = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=['Actual: No cardiovascular disease', 'Actual: Cardiovascular disease'],
    columns=['Predicted: No cardiovascular disease', 'Predicted: Cardiovascular disease'],
)
confusion_df.to_csv(TABLES_DIR / 'model_06_confusion_matrix_test.csv')

display(confusion_df)

fig, ax = plt.subplots(figsize=(7, 5.5))
image = ax.imshow(confusion_df.values, cmap='Blues')

ax.set_xticks(np.arange(2))
ax.set_yticks(np.arange(2))
ax.set_xticklabels(['Predicted No', 'Predicted Yes'])
ax.set_yticklabels(['Actual No', 'Actual Yes'])
ax.set_title('Final test confusion matrix')

for row in range(2):
    for column in range(2):
        value = confusion_df.values[row, column]
        text_colour = 'white' if value > confusion_df.values.max() * 0.55 else 'black'
        ax.text(column, row, f'{value:,}', ha='center', va='center', color=text_colour, fontsize=13, fontweight='bold')

ax.set_xlabel('Predicted class')
ax.set_ylabel('Actual class')
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

confusion_chart_path = CHARTS_DIR / 'model_02_confusion_matrix_test.png'
fig.tight_layout()
fig.savefig(confusion_chart_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved chart: {confusion_chart_path}')

## 12. ROC Curve

ROC-AUC measures how well the model ranks positive cases above negative cases across all possible thresholds.

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, test_probabilities)
final_roc_auc = roc_auc_score(y_test, test_probabilities)

roc_curve_df = pd.DataFrame({
    'false_positive_rate': fpr,
    'true_positive_rate': tpr,
    'threshold': roc_thresholds,
})
roc_curve_df.to_csv(TABLES_DIR / 'model_07_roc_curve_points_test.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#2F4858', linewidth=2.2, label=f'ROC curve, AUC = {final_roc_auc:.3f}')
ax.plot([0, 1], [0, 1], color='#8A8F98', linestyle='--', linewidth=1.5, label='No-skill baseline')
ax.set_title('Final test ROC curve')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate / Recall')
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(color='#D9DEE7', linewidth=0.8)
ax.legend(loc='lower right')

roc_chart_path = CHARTS_DIR / 'model_03_roc_curve_test.png'
fig.tight_layout()
fig.savefig(roc_chart_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved chart: {roc_chart_path}')

## 13. Precision-Recall Curve

This curve is useful because the project has a screening-support use case. It shows the trade-off between catching more positive cases and keeping positive predictions reliable.

In [ ]:
precision_values, recall_values, pr_thresholds = precision_recall_curve(y_test, test_probabilities)
final_average_precision = average_precision_score(y_test, test_probabilities)

precision_recall_df = pd.DataFrame({
    'precision': precision_values,
    'recall': recall_values,
})
precision_recall_df.to_csv(TABLES_DIR / 'model_08_precision_recall_curve_points_test.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(recall_values, precision_values, color='#2F4858', linewidth=2.2, label=f'Average precision = {final_average_precision:.3f}')
ax.axhline(y_test.mean(), color='#8A8F98', linestyle='--', linewidth=1.5, label='Positive class rate')
ax.set_title('Final test precision-recall curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(color='#D9DEE7', linewidth=0.8)
ax.legend(loc='lower left')

pr_chart_path = CHARTS_DIR / 'model_04_precision_recall_curve_test.png'
fig.tight_layout()
fig.savefig(pr_chart_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved chart: {pr_chart_path}')

## 14. Predicted Probability Distribution

This chart shows whether the model separates the two classes. Stronger separation means the model assigns higher probabilities to positive cases more often.

In [ ]:
probability_plot_df = pd.DataFrame({
    'actual_cardio': y_test.values,
    'predicted_probability': test_probabilities,
})
probability_plot_df['actual_label'] = probability_plot_df['actual_cardio'].map({0: 'No cardiovascular disease', 1: 'Cardiovascular disease'})
probability_plot_df.to_csv(TABLES_DIR / 'model_09_test_predicted_probabilities.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
for label, colour in [('No cardiovascular disease', '#2F4858'), ('Cardiovascular disease', '#F26419')]:
    subset = probability_plot_df.loc[probability_plot_df['actual_label'] == label, 'predicted_probability']
    ax.hist(subset, bins=30, alpha=0.55, density=True, label=label, color=colour)

ax.axvline(selected_threshold, color='black', linestyle='--', linewidth=1.8, label=f'Selected threshold: {selected_threshold:.2f}')
ax.set_title('Predicted probability distribution on final test set')
ax.set_xlabel('Predicted probability of cardiovascular disease')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis='y', color='#D9DEE7', linewidth=0.8)
ax.legend(loc='upper center')

probability_chart_path = CHARTS_DIR / 'model_05_probability_distribution_test.png'
fig.tight_layout()
fig.savefig(probability_chart_path, dpi=180, bbox_inches='tight')
plt.show()
print(f'Saved chart: {probability_chart_path}')

## 15. Feature Importance

Permutation importance shows how much model performance falls when a field is shuffled.

This is easier to explain than raw coefficients here because several health fields are related to each other, such as systolic blood pressure, diastolic blood pressure, pulse pressure, and blood pressure bands.

Higher importance means the model relied more on that field for ranking cardiovascular disease risk on the test set. This still shows association, not causation.

In [ ]:
selected_input_features = (
    selected_candidate_config['numeric_features']
    + selected_candidate_config['categorical_features']
)

permutation_result = permutation_importance(
    final_model,
    test_df[selected_input_features],
    y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=None,
)

feature_effects = pd.DataFrame({
    'feature': selected_input_features,
    'importance_mean': permutation_result.importances_mean,
    'importance_std': permutation_result.importances_std,
})

feature_label_map = {
    'age_years': 'Age',
    'bmi': 'BMI',
    'ap_hi': 'Systolic BP',
    'ap_lo': 'Diastolic BP',
    'pulse_pressure': 'Pulse pressure',
    'mean_arterial_pressure': 'Mean arterial pressure',
    'gender': 'Gender code',
    'age_band': 'Age band',
    'age_decade': 'Age decade',
    'bmi_category': 'BMI category',
    'systolic_bp_category': 'Systolic BP category',
    'diastolic_bp_category': 'Diastolic BP category',
    'pulse_pressure_band': 'Pulse pressure band',
    'cholesterol': 'Cholesterol',
    'gluc': 'Glucose',
    'smoke': 'Smoking',
    'alco': 'Alcohol intake',
    'active': 'Physical activity',
}
feature_effects['feature_label'] = feature_effects['feature'].map(feature_label_map).fillna(feature_effects['feature'])
feature_effects = feature_effects.sort_values('importance_mean', ascending=False).reset_index(drop=True)
feature_effects.to_csv(TABLES_DIR / 'model_10_feature_effects.csv', index=False)

plot_effects = feature_effects.head(15).sort_values('importance_mean')

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_effects['feature_label'], plot_effects['importance_mean'], color='#2F4858')
ax.set_title('Permutation importance on final test set')
ax.set_xlabel('Mean ROC-AUC decrease when shuffled')
ax.set_ylabel('Input feature')
ax.grid(axis='x', color='#D9DEE7', linewidth=0.8)

feature_chart_path = CHARTS_DIR / 'model_06_feature_effects.png'
fig.tight_layout()
fig.savefig(feature_chart_path, dpi=180, bbox_inches='tight')
plt.show()

rounded_feature_effects = feature_effects.copy()
rounded_feature_effects['importance_mean'] = rounded_feature_effects['importance_mean'].round(5)
rounded_feature_effects['importance_std'] = rounded_feature_effects['importance_std'].round(5)

display(rounded_feature_effects)
print(f'Saved feature importance table: {TABLES_DIR / "model_10_feature_effects.csv"}')
print(f'Saved chart: {feature_chart_path}')

## 16. Model Evidence Summary

This cell creates a short table that can be reused later in the portfolio or project write-up.

In [ ]:
model_evidence_summary = pd.DataFrame([
    {
        'item': 'Research question',
        'summary': 'To what extent can routine health indicators be used to predict whether an individual has cardiovascular disease?'
    },
    {
        'item': 'Use case',
        'summary': 'Cardiovascular risk-screening prototype for prioritisation and communication, not diagnosis.'
    },
    {
        'item': 'Selected model',
        'summary': selected_candidate_name
    },
    {
        'item': 'Selected threshold',
        'summary': f'{selected_threshold:.2f}'
    },
    {
        'item': 'Final test accuracy',
        'summary': f'{final_test_metrics.loc[0, "accuracy"]:.1%}'
    },
    {
        'item': 'Final test precision',
        'summary': f'{final_test_metrics.loc[0, "precision"]:.1%}'
    },
    {
        'item': 'Final test recall',
        'summary': f'{final_test_metrics.loc[0, "recall"]:.1%}'
    },
    {
        'item': 'Final test F1 score',
        'summary': f'{final_test_metrics.loc[0, "f1"]:.1%}'
    },
    {
        'item': 'Final test ROC-AUC',
        'summary': f'{final_test_metrics.loc[0, "roc_auc"]:.3f}'
    },
    {
        'item': 'Most serious error',
        'summary': 'False negative: actual cardiovascular disease predicted as no cardiovascular disease.'
    },
    {
        'item': 'Important limitation',
        'summary': 'The dataset supports prediction and association, not medical diagnosis or causal claims.'
    },
])

model_evidence_summary.to_csv(TABLES_DIR / 'model_11_evidence_summary.csv', index=False)
display(model_evidence_summary)
print(f'Saved evidence summary to: {TABLES_DIR / "model_11_evidence_summary.csv"}')

## 17. Final Checks

These checks confirm that the modelling evidence files were created.

In [ ]:
required_tables = [
    TABLES_DIR / 'model_00_train_validation_test_split_summary.csv',
    TABLES_DIR / 'model_01_candidate_comparison_validation.csv',
    TABLES_DIR / 'model_02_all_candidate_threshold_scans_validation.csv',
    TABLES_DIR / 'model_03_selected_model_summary.csv',
    TABLES_DIR / 'model_04_selected_threshold_scan_validation.csv',
    TABLES_DIR / 'model_05_final_test_metrics.csv',
    TABLES_DIR / 'model_06_confusion_matrix_test.csv',
    TABLES_DIR / 'model_07_roc_curve_points_test.csv',
    TABLES_DIR / 'model_08_precision_recall_curve_points_test.csv',
    TABLES_DIR / 'model_09_test_predicted_probabilities.csv',
    TABLES_DIR / 'model_10_feature_effects.csv',
    TABLES_DIR / 'model_11_evidence_summary.csv',
]

required_charts = [
    CHARTS_DIR / 'model_01_validation_threshold_tradeoff.png',
    CHARTS_DIR / 'model_02_confusion_matrix_test.png',
    CHARTS_DIR / 'model_03_roc_curve_test.png',
    CHARTS_DIR / 'model_04_precision_recall_curve_test.png',
    CHARTS_DIR / 'model_05_probability_distribution_test.png',
    CHARTS_DIR / 'model_06_feature_effects.png',
]

missing_files = [path for path in [*required_tables, *required_charts] if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing expected modelling outputs: {missing_files}')

assert final_test_metrics.loc[0, 'roc_auc'] > 0.75, 'ROC-AUC should show useful predictive signal.'
assert final_test_metrics.loc[0, 'recall'] > 0.75, 'Recall should remain strong for the screening-support use case.'
assert final_test_metrics.loc[0, 'f1'] > 0.70, 'F1 score should remain acceptable for the chosen threshold.'

print('Final checks passed.')
print(f'Selected model: {selected_candidate_name}')
print(f'Selected threshold: {selected_threshold:.2f}')
print(f'Final test ROC-AUC: {final_test_metrics.loc[0, "roc_auc"]:.3f}')
print(f'Final test recall: {final_test_metrics.loc[0, "recall"]:.1%}')
print(f'Final test F1 score: {final_test_metrics.loc[0, "f1"]:.1%}')

## Next Step

Review the model metrics, confusion matrix, ROC curve, precision-recall curve, and feature effects.

After review, the next step is to decide whether the selected model and threshold should be used for the prototype app.